Analyse der Excel

Erster Schritt neue Spalten pro Hobby einfuegen

In [1]:
import re
import pandas as pd
from pathlib import Path

FILENAME = "Lets Meet DB Dump.xlsx"
OUTPUT_NAME = "Let_Meet_Hobby_Col.xlsx"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # __file__ existiert nicht in Jupyter-Notebook-Zellen -> aktuelles Arbeitsverzeichnis nutzen
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    """Sucht die Excel-Datei an mehreren plausiblen Orten und gibt den ersten Treffer zurück."""
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad

    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]

    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}\n"
        f"Bitte Datei manuell in eines dieser Verzeichnisse legen oder Pfad direkt angeben."
    )


XLSX_PATH = finde_xlsx()
OUTPUT_PATH = XLSX_PATH.parent / OUTPUT_NAME

HOBBY_COL = "Hobby1 %Prio1%; Hobby2 %Prio2%; Hobby3 %Prio3%; Hobby4 %Prio4%; Hobby5 %Prio5%;"
HOBBY_PATTERN = re.compile(r"([^%;]+?)\s*%(\d+)%\s*;?")
MAX_HOBBYS = 5


def main():
    df = pd.read_excel(XLSX_PATH)

    print(f"Eingelesen: {XLSX_PATH} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")
    print("\nVorher (erste 3 Zeilen der Hobby-Spalte):")
    print(df[HOBBY_COL].head(3).to_string())

    # Für jede Zeile bis zu 5 (Hobby, Prio)-Paare extrahieren
    parsed = df[HOBBY_COL].apply(
        lambda text: HOBBY_PATTERN.findall(text) if pd.notna(text) else []
    )

    # Neue Spalten Hobby1..Hobby5, Prio1..Prio5 befüllen
    for i in range(1, MAX_HOBBYS + 1):
        df[f"Hobby{i}"] = parsed.apply(
            lambda paare, i=i: paare[i - 1][0].strip() if len(paare) >= i else pd.NA
        )
        df[f"Prio{i}"] = parsed.apply(
            lambda paare, i=i: int(paare[i - 1][1]) if len(paare) >= i else pd.NA
        )

    print(f"\nNeue Spalten hinzugefügt: {[f'Hobby{i}' for i in range(1, MAX_HOBBYS + 1)]} "
          f"+ {[f'Prio{i}' for i in range(1, MAX_HOBBYS + 1)]}")

    print("\nNachher (erste 3 Zeilen, neue Spalten):")
    neue_spalten = [f"Hobby{i}" for i in range(1, MAX_HOBBYS + 1)] + \
                   [f"Prio{i}" for i in range(1, MAX_HOBBYS + 1)]
    print(df[neue_spalten].head(3).to_string())

    df.to_excel(OUTPUT_PATH, index=False)
    print(f"\nGespeichert: {OUTPUT_PATH} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")
    print("Originaldatei wurde nicht verändert.")


if __name__ == "__main__":
    main()

<jemalloc>: Unsupported system page size


Eingelesen: /home/jovyan/LetsMeet/Lets Meet DB Dump.xlsx (1576 Zeilen, 8 Spalten)

Vorher (erste 3 Zeilen der Hobby-Spalte):
0    Fremdsprachenkenntnisse erweitern %78%; Im Was...
1    Für jemanden kochen %57%; Mir die Probleme von...
2    In der Stadt herumbummeln %94%; Mit Freunden z...

Neue Spalten hinzugefügt: ['Hobby1', 'Hobby2', 'Hobby3', 'Hobby4', 'Hobby5'] + ['Prio1', 'Prio2', 'Prio3', 'Prio4', 'Prio5']

Nachher (erste 3 Zeilen, neue Spalten):
                              Hobby1                                Hobby2                                               Hobby3                         Hobby4 Hobby5 Prio1 Prio2 Prio3 Prio4 Prio5
0  Fremdsprachenkenntnisse erweitern                       Im Wasser waten                           Schwierige Probleme klären         Morgens Früh aufstehen   <NA>    78    80    61    17  <NA>
1                Für jemanden kochen  Mir die Probleme von anderen anhören  Abends seinem Partner Ereignisse des Tages erzählen  Für einen guten Zweck 

In [2]:
import re
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_Hobby_Col.xlsx"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}\n"
        f"Bitte zuerst step1_hobby_spalten.py ausführen."
    )


XLSX_PATH = finde_xlsx()
HOBBY_SPALTEN = [f"Hobby{i}" for i in range(1, 6)]
PRIO_SPALTEN = [f"Prio{i}" for i in range(1, 6)]
ORIG_COL = "Hobby1 %Prio1%; Hobby2 %Prio2%; Hobby3 %Prio3%; Hobby4 %Prio4%; Hobby5 %Prio5%;"
HOBBY_PATTERN = re.compile(r"([^%;]+?)\s*%(\d+)%\s*;?")


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    df = pd.read_excel(XLSX_PATH)
    print(f"Eingelesen: {XLSX_PATH}")
    print(f"Zeilen: {df.shape[0]}, Spalten: {df.shape[1]}")

    fehlende_spalten = [c for c in HOBBY_SPALTEN + PRIO_SPALTEN if c not in df.columns]
    if fehlende_spalten:
        raise ValueError(f"Erwartete Spalten fehlen: {fehlende_spalten}")

    # ------------------------------------------------------------
    section("1. Stichprobe: erste 5 Zeilen (nur Hobby-Spalten)")
    print(df[["Nachname, Vorname"] + HOBBY_SPALTEN + PRIO_SPALTEN].head(5).to_string())

    # ------------------------------------------------------------
    section("2. Fehlende Werte pro Hobby-/Prio-Spalte")
    missing = df[HOBBY_SPALTEN + PRIO_SPALTEN].isna().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    report = pd.DataFrame({"fehlend": missing, "anteil_%": missing_pct})
    print(report)
    print("\nHinweis: Hobby5/Prio5 haben erwartungsgemäß am meisten fehlende Werte,")
    print("da nicht jeder Nutzer 5 Hobbys angegeben hat.")

    # ------------------------------------------------------------
    section("3. Wertebereich der Prioritäten (sollte 0-100 sein)")
    for col in PRIO_SPALTEN:
        werte = df[col].dropna()
        if len(werte) == 0:
            continue
        print(f"{col}: min={werte.min()}, max={werte.max()}, n={len(werte)}")
        ausserhalb = werte[(werte < 0) | (werte > 100)]
        if len(ausserhalb) > 0:
            print(f"  ACHTUNG: {len(ausserhalb)} Werte außerhalb 0-100!")

    # ------------------------------------------------------------
    section("4. Konsistenz: Hobby-Spalte vorhanden aber Prio fehlt (oder umgekehrt)")
    inkonsistent_gesamt = 0
    for hobby_col, prio_col in zip(HOBBY_SPALTEN, PRIO_SPALTEN):
        hobby_da = df[hobby_col].notna()
        prio_da = df[prio_col].notna()
        inkonsistent = df[hobby_da != prio_da]
        if len(inkonsistent) > 0:
            inkonsistent_gesamt += len(inkonsistent)
            print(f"{hobby_col}/{prio_col}: {len(inkonsistent)} inkonsistente Zeilen")
            print(inkonsistent[["Nachname, Vorname", hobby_col, prio_col]].head(5).to_string(index=False))
    if inkonsistent_gesamt == 0:
        print("Keine Inkonsistenzen gefunden – jede vorhandene Hobby-Angabe hat eine Priorität und umgekehrt.")

    # ------------------------------------------------------------
    section("5. Duplikate: gleiches Hobby mehrfach in einer Zeile")
    def hat_duplikate(row):
        hobbys = [row[c] for c in HOBBY_SPALTEN if pd.notna(row[c])]
        return len(hobbys) != len(set(hobbys))

    dup_mask = df.apply(hat_duplikate, axis=1)
    print(f"Zeilen mit doppelt genanntem Hobby: {dup_mask.sum()}")
    if dup_mask.sum() > 0:
        print(df[dup_mask][["Nachname, Vorname"] + HOBBY_SPALTEN].head(5).to_string(index=False))

    # ------------------------------------------------------------
    section("6. Rückwärts-Konsistenzprüfung gegen Originalspalte")
    print("Vergleicht für eine Stichprobe: aus Hobby1-5/Prio1-5 rekonstruierter Text")
    print("vs. beim erneuten Parsen der Originalspalte extrahierte Werte.\n")

    abweichungen = 0
    stichprobe = df.sample(min(50, len(df)), random_state=42)
    for idx, row in stichprobe.iterrows():
        orig_text = row[ORIG_COL]
        orig_paare = HOBBY_PATTERN.findall(orig_text) if pd.notna(orig_text) else []
        orig_paare = [(h.strip(), int(p)) for h, p in orig_paare]

        neu_paare = []
        for hobby_col, prio_col in zip(HOBBY_SPALTEN, PRIO_SPALTEN):
            if pd.notna(row[hobby_col]):
                neu_paare.append((row[hobby_col], int(row[prio_col])))

        if orig_paare != neu_paare:
            abweichungen += 1
            if abweichungen <= 5:
                print(f"Zeile {idx} ({row['Nachname, Vorname']}): Abweichung")
                print(f"  Original: {orig_paare}")
                print(f"  Neu:      {neu_paare}")

    print(f"\nGeprüfte Stichprobe: {len(stichprobe)} Zeilen")
    print(f"Abweichungen gefunden: {abweichungen}")
    if abweichungen == 0:
        print("Alle geprüften Zeilen stimmen exakt mit der Originalspalte überein.")

    # ------------------------------------------------------------
    section("7. Häufigste Hobbys über alle 5 Spalten hinweg")
    alle_hobbys = pd.concat([df[c].dropna() for c in HOBBY_SPALTEN])
    print(f"Anzahl Hobby-Nennungen gesamt: {len(alle_hobbys)}")
    print(f"Anzahl unterschiedlicher Hobbys: {alle_hobbys.nunique()}")
    print("\nTop 10:")
    print(alle_hobbys.value_counts().head(10))

    # ------------------------------------------------------------
    section("Ende der Hobby-Spalten-Prüfung")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_Hobby_Col.xlsx
Zeilen: 1576, Spalten: 18

1. Stichprobe: erste 5 Zeilen (nur Hobby-Spalten)
    Nachname, Vorname                             Hobby1                                     Hobby2                                               Hobby3                                               Hobby4 Hobby5  Prio1  Prio2  Prio3  Prio4  Prio5
0     Forster, Martin  Fremdsprachenkenntnisse erweitern                            Im Wasser waten                           Schwierige Probleme klären                               Morgens Früh aufstehen    NaN   78.0   80.0   61.0   17.0    NaN
1  Elina, Tsanaklidou                Für jemanden kochen       Mir die Probleme von anderen anhören  Abends seinem Partner Ereignisse des Tages erzählen                        Für einen guten Zweck spenden    NaN   57.0   21.0   53.0   25.0    NaN
2       Vildan, şahin          In der Stadt herumbummeln                 Mit Freunden zusammen sein                      

Email Ueberpruefung

In [3]:
import pandas as pd
from pathlib import Path

FILENAME = "Lets Meet DB Dump.xlsx"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


XLSX_PATH = finde_xlsx()
EMAIL_COL = "E-Mail"


def main():
    df = pd.read_excel(XLSX_PATH)
    print(f"Eingelesen: {XLSX_PATH} ({df.shape[0]} Zeilen)")

    # Alles ab und inklusive '@' extrahieren
    df["Email_Domain"] = df[EMAIL_COL].str.extract(r"(@.+)$")

    fehlend = df["Email_Domain"].isna().sum()
    print(f"E-Mails ohne '@' bzw. leer: {fehlend}")

    print(f"\nGesamtzahl E-Mail-Adressen: {df[EMAIL_COL].notna().sum()}")
    print(f"Anzahl unterschiedlicher Domains (inkl. '@'): {df['Email_Domain'].nunique()}")

    domain_counts = df["Email_Domain"].value_counts()

    print(domain_counts.to_string())

    print(f"\nSumme Kontrolle: {domain_counts.sum()} (sollte {df[EMAIL_COL].notna().sum()} entsprechen)")

    # Export als CSV für Weiterverwendung
    out_path = XLSX_PATH.parent / "email_domain_counts.csv"
    domain_counts.rename("Anzahl").rename_axis("Domain").to_csv(out_path)
    print(f"\nGespeichert: {out_path}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Lets Meet DB Dump.xlsx (1576 Zeilen)
E-Mails ohne '@' bzw. leer: 0

Gesamtzahl E-Mail-Adressen: 1576
Anzahl unterschiedlicher Domains (inkl. '@'): 24
Email_Domain
@d-ohnline.te      166
@gmaiil.te         158
@web.te            122
@ge-em-ix.te       107
@gmaiil.ork         84
@autluuk.te         83
@d-ohnline.ork      81
@gmaiil.kom         79
@d-ohnline.kom      78
@web.ork            75
@web.kom            67
@iunitimail.te      62
@ge-em-ix.ork       61
@ge-em-ix.kom       47
@autluuk.ork        41
@1mal1.te           38
@a-o-l.te           35
@autluuk.kom        34
@iunitimail.kom     33
@iunitimail.ork     32
@a-o-l.kom          30
@a-o-l.ork          26
@1mal1.kom          23
@1mal1.ork          14

Summe Kontrolle: 1576 (sollte 1576 entsprechen)

Gespeichert: /home/jovyan/LetsMeet/email_domain_counts.csv


zwischen @ und .

In [4]:
import pandas as pd
from pathlib import Path

DOMAIN_MAPPING = {
    "d-ohnline": "t-online",
    "gmaiil": "gmail",
    "web": "web",
    "ge-em-ix": "gmx",
    "autluuk": "outlook",
    "iunitimail": "iunitimail",
    "a-o-l": "aol",
    "1mal1": "1und1",   
}

TLD_MAPPING = {
    "te": "de",
    "ork": "org",
    "kom": "com",
}

INPUT_FILENAME = "Let_Meet_Hobby_Col.xlsx"
OUTPUT_FILENAME = "Let_Meet_H_Email.xlsx"
EMAIL_COL = "E-Mail"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_datei(dateiname):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


def main():
    input_path = finde_datei(INPUT_FILENAME)
    output_path = input_path.parent / OUTPUT_FILENAME

    df = pd.read_excel(input_path)
    print(f"Eingelesen: {input_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

    print("\nVorher (erste 5 E-Mails):")
    print(df[EMAIL_COL].head(5).to_string())

    # Domain-Name (zwischen @ und erstem Punkt) und Endung (nach letztem Punkt) extrahieren
    domain_name = df[EMAIL_COL].str.extract(r"@([^.]+)\.")[0]
    tld = df[EMAIL_COL].str.extract(r"\.([^.]+)$")[0]
    lokal_teil = df[EMAIL_COL].str.extract(r"^([^@]+)@")[0]

    unbekannte_namen = set(domain_name.dropna().unique()) - set(DOMAIN_MAPPING.keys())
    if unbekannte_namen:
        print(f"\nWARNUNG: Folgende Domain-Namen sind NICHT in DOMAIN_MAPPING enthalten "
              f"und bleiben unverändert: {sorted(unbekannte_namen)}")

    unbekannte_tlds = set(tld.dropna().unique()) - set(TLD_MAPPING.keys())
    if unbekannte_tlds:
        print(f"WARNUNG: Folgende Endungen sind NICHT in TLD_MAPPING enthalten "
              f"und bleiben unverändert: {sorted(unbekannte_tlds)}")

    neuer_name = domain_name.map(DOMAIN_MAPPING).fillna(domain_name)
    neue_tld = tld.map(TLD_MAPPING).fillna(tld)
    df[EMAIL_COL] = lokal_teil + "@" + neuer_name + "." + neue_tld

    print("\nNachher (erste 5 E-Mails):")
    print(df[EMAIL_COL].head(5).to_string())

    print("\nDomain-Name-Zuordnung angewendet:")
    for alt, neu in DOMAIN_MAPPING.items():
        anzahl = (domain_name == alt).sum()
        print(f"  {alt:15s} -> {neu:15s} ({anzahl} Einträge)")

    print("\nEndungs-Zuordnung angewendet:")
    for alt, neu in TLD_MAPPING.items():
        anzahl = (tld == alt).sum()
        print(f"  .{alt:5s} -> .{neu:5s} ({anzahl} Einträge)")

    df.to_excel(output_path, index=False)
    print(f"\nGespeichert: {output_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")
    print(f"Eingabedatei ({input_path.name}) wurde nicht verändert.")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_Hobby_Col.xlsx (1576 Zeilen, 18 Spalten)

Vorher (erste 5 E-Mails):
0          martin.forster@web.ork
1      tsanaklidou.elina@1mal1.te
2         şahin.vildan@gmaiil.ork
3           ellen.bäumker@web.ork
4    bekiroğlu.bahadır@autluuk.te

Nachher (erste 5 E-Mails):
0          martin.forster@web.org
1      tsanaklidou.elina@1und1.de
2          şahin.vildan@gmail.org
3           ellen.bäumker@web.org
4    bekiroğlu.bahadır@outlook.de

Domain-Name-Zuordnung angewendet:
  d-ohnline       -> t-online        (325 Einträge)
  gmaiil          -> gmail           (321 Einträge)
  web             -> web             (264 Einträge)
  ge-em-ix        -> gmx             (215 Einträge)
  autluuk         -> outlook         (158 Einträge)
  iunitimail      -> iunitimail      (127 Einträge)
  a-o-l           -> aol             (91 Einträge)
  1mal1           -> 1und1           (75 Einträge)

Endungs-Zuordnung angewendet:
  .te    -> .de    (771 Einträge)
  .ork 

In [5]:
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_H_Email.xlsx"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


XLSX_PATH = finde_xlsx()
GESCHLECHT_COL = "Geschlecht (m/w/nonbinary)"
INTERESSE_COL = "Interessiert an"


def main():
    df = pd.read_excel(XLSX_PATH)
    print(f"Eingelesen: {XLSX_PATH} ({df.shape[0]} Zeilen)")

    print(f"\n{GESCHLECHT_COL}:")
    print(df[GESCHLECHT_COL].value_counts())
    print(f"Summe: {df[GESCHLECHT_COL].value_counts().sum()}")

    print(f"\n{INTERESSE_COL}:")
    print(df[INTERESSE_COL].value_counts())
    print(f"Summe: {df[INTERESSE_COL].value_counts().sum()}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen)

Geschlecht (m/w/nonbinary):
Geschlecht (m/w/nonbinary)
m     918
w     620
nb     38
Name: count, dtype: int64
Summe: 1576

Interessiert an:
Interessiert an
w     908
m     635
mw     33
Name: count, dtype: int64
Summe: 1576


In [6]:
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_H_Email.xlsx"
STICHWORT = "Geburtsdatum"
REFERENZDATUM = pd.Timestamp("2026-08-17")

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


def finde_spalte(df, stichwort):
    treffer = [c for c in df.columns if stichwort.lower() in c.lower()]
    if not treffer:
        raise KeyError(
            f"Keine Spalte gefunden, die '{stichwort}' enthält. "
            f"Vorhandene Spalten: {list(df.columns)}"
        )
    if len(treffer) > 1:
        print(f"WARNUNG: Mehrere Spalten passen zu '{stichwort}': {treffer}. Nehme die erste.")
    return treffer[0]


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    xlsx_path = finde_xlsx()
    df = pd.read_excel(xlsx_path)
    print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen)")

    col = finde_spalte(df, STICHWORT)
    print(f"Verwendete Spalte: {col!r}")

    section("1. Stichprobe (erste 10 Werte, Rohformat)")
    print(df[col].head(10).to_string())

    section("2. Fehlende Werte")
    fehlend = df[col].isna().sum()
    print(f"Fehlende Geburtsdaten: {fehlend} von {len(df)}")

    section("3. Parsebarkeit (Format TT.MM.JJJJ)")
    geburtsdatum = pd.to_datetime(df[col], format="%d.%m.%Y", errors="coerce")
    nicht_parsebar = df[geburtsdatum.isna() & df[col].notna()]
    print(f"Nicht im Format TT.MM.JJJJ parsebar: {len(nicht_parsebar)}")
    if len(nicht_parsebar) > 0:
        print(nicht_parsebar[[col]].head(15).to_string(index=False))

    section("4. Zukünftige Geburtsdaten")
    zukunft = df[geburtsdatum > REFERENZDATUM]
    print(f"Geburtsdatum nach {REFERENZDATUM.date()}: {len(zukunft)}")
    if len(zukunft) > 0:
        print(zukunft[[col]].head(10).to_string(index=False))

    section("5. Alter berechnen & Plausibilität prüfen")
    alter = ((REFERENZDATUM - geburtsdatum).dt.days // 365.25)
    print(f"Min. Alter: {alter.min()}")
    print(f"Max. Alter: {alter.max()}")
    print(f"Durchschnittsalter: {alter.mean():.1f}")
    print(f"Median Alter: {alter.median():.0f}")

    unplausibel = df[(alter < 18) | (alter > 100)]
    print(f"\nUnplausibles Alter (<18 oder >100 Jahre): {len(unplausibel)}")
    if len(unplausibel) > 0:
        anzeige = unplausibel[[col]].copy()
        anzeige["Alter"] = alter[unplausibel.index]
        print(anzeige.head(15).to_string(index=False))

    section("6. Verteilung nach Geburtsjahrzehnt")
    jahrzehnt = (geburtsdatum.dt.year // 10 * 10).astype("Int64")
    print(jahrzehnt.value_counts().sort_index())

    section("7. Zusammenfassung")
    print(f"Zeilen gesamt: {len(df)}")
    print(f"Gültig geparst: {geburtsdatum.notna().sum()}")
    print(f"Fehlend: {fehlend}")
    print(f"Nicht parsebar: {len(nicht_parsebar)}")
    print(f"In der Zukunft: {len(zukunft)}")
    print(f"Unplausibles Alter: {len(unplausibel)}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen)
Verwendete Spalte: 'Geburtsdatum'

1. Stichprobe (erste 10 Werte, Rohformat)
0    07.03.1959
1    28.02.1958
2    27.07.1996
3    09.12.1992
4    13.06.1983
5    07.08.1980
6    18.04.1983
7    19.01.1967
8    21.02.1968
9    30.07.1995

2. Fehlende Werte
Fehlende Geburtsdaten: 0 von 1576

3. Parsebarkeit (Format TT.MM.JJJJ)
Nicht im Format TT.MM.JJJJ parsebar: 0

4. Zukünftige Geburtsdaten
Geburtsdatum nach 2026-08-17: 0

5. Alter berechnen & Plausibilität prüfen
Min. Alter: 23.0
Max. Alter: 68.0
Durchschnittsalter: 45.4
Median Alter: 46

Unplausibles Alter (<18 oder >100 Jahre): 0

6. Verteilung nach Geburtsjahrzehnt
Geburtsdatum
1950     64
1960    343
1970    361
1980    360
1990    340
2000    108
Name: count, dtype: Int64

7. Zusammenfassung
Zeilen gesamt: 1576
Gültig geparst: 1576
Fehlend: 0
Nicht parsebar: 0
In der Zukunft: 0
Unplausibles Alter: 0


In [7]:
"""
LetsMeet - Telefonnummer überprüfen
====================================================================
Liest Let_Meet_H_Email.xlsx ein und prüft die Spalte 'Telefon' auf
Format-Konsistenz und Plausibilität:
  - Erlaubte Zeichen (Ziffern, Leerzeichen, /, (), -)
  - Strukturmuster (wie uneinheitlich ist das Format?)
  - Gesamtzahl der Ziffern (deutsche Rufnummern: i.d.R. 10-12)
  - Vorwahl-Plausibilität (deutsche Vorwahlen beginnen mit 0)
  - Duplikate
  - Fehlende Werte

Aufruf: python3 step7_telefon_pruefen.py
"""

import re
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_H_Email.xlsx"
STICHWORT = "Telefon"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


def finde_spalte(df, stichwort):
    treffer = [c for c in df.columns if stichwort.lower() in c.lower()]
    if not treffer:
        raise KeyError(
            f"Keine Spalte gefunden, die '{stichwort}' enthält. "
            f"Vorhandene Spalten: {list(df.columns)}"
        )
    if len(treffer) > 1:
        print(f"WARNUNG: Mehrere Spalten passen zu '{stichwort}': {treffer}. Nehme die erste.")
    return treffer[0]


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    xlsx_path = finde_xlsx()
    df = pd.read_excel(xlsx_path)
    print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen)")

    col = finde_spalte(df, STICHWORT)
    print(f"Verwendete Spalte: {col!r}")

    section("1. Stichprobe (erste 10 Werte, Rohformat)")
    print(df[col].head(10).to_string())

    section("2. Fehlende Werte")
    fehlend = df[col].isna().sum()
    print(f"Fehlende Telefonnummern: {fehlend} von {len(df)}")

    section("3. Erlaubte Zeichen prüfen")
    muster_erlaubt = re.compile(r"^[\d\s()/\-+.]+$")
    ungueltig = df[~df[col].astype(str).apply(lambda v: bool(muster_erlaubt.match(v)))]
    print(f"Nummern mit unerwarteten Zeichen: {len(ungueltig)}")
    if len(ungueltig) > 0:
        print(ungueltig[[col]].head(15).to_string(index=False))

    section("4. Strukturmuster (Ziffern durch # ersetzt)")
    def struktur(t):
        return re.sub(r"\d", "#", str(t))
    muster_counts = df[col].apply(struktur).value_counts()
    print(f"Anzahl unterschiedlicher Strukturmuster: {len(muster_counts)}")
    print(muster_counts.to_string())

    section("5. Anzahl Ziffern je Nummer")
    anzahl_ziffern = df[col].astype(str).apply(lambda t: len(re.sub(r"\D", "", t)))
    print(anzahl_ziffern.describe())
    print("\nVerteilung:")
    print(anzahl_ziffern.value_counts().sort_index())

    zu_kurz = df[anzahl_ziffern < 9]
    zu_lang = df[anzahl_ziffern > 13]
    print(f"\nVermutlich zu kurz (<9 Ziffern): {len(zu_kurz)}")
    if len(zu_kurz) > 0:
        print(zu_kurz[["Nachname, Vorname", col]].to_string(index=False))
    print(f"Vermutlich zu lang (>13 Ziffern): {len(zu_lang)}")
    if len(zu_lang) > 0:
        print(zu_lang[["Nachname, Vorname", col]].to_string(index=False))

    section("6. Vorwahl-Plausibilität (sollte mit 0 beginnen)")
    erste_ziffer = df[col].astype(str).str.strip().str.lstrip("(").str[0]
    nicht_null = df[erste_ziffer != "0"]
    print(f"Nummern, die nicht mit '0' beginnen: {len(nicht_null)}")
    if len(nicht_null) > 0:
        print(nicht_null[[col]].head(15).to_string(index=False))

    section("7. Duplikate")
    # Normalisiert vergleichen (nur Ziffern), um Formatunterschiede zu ignorieren
    nur_ziffern = df[col].astype(str).apply(lambda t: re.sub(r"\D", "", t))
    dup_mask = nur_ziffern.duplicated(keep=False)
    print(f"Exakt gleiche Rohwerte: {df[col].duplicated().sum()}")
    print(f"Gleiche Nummer bei unterschiedlicher Schreibweise (nur Ziffern verglichen): {dup_mask.sum()}")
    if dup_mask.sum() > 0:
        print(df[dup_mask][["Nachname, Vorname", col]].sort_values(col).head(15).to_string(index=False))

    section("8. Zusammenfassung")
    print(f"Zeilen gesamt: {len(df)}")
    print(f"Fehlend: {fehlend}")
    print(f"Unerwartete Zeichen: {len(ungueltig)}")
    print(f"Unterschiedliche Strukturmuster: {len(muster_counts)}")
    print(f"Zu kurz (<9 Ziffern): {len(zu_kurz)}")
    print(f"Zu lang (>13 Ziffern): {len(zu_lang)}")
    print(f"Beginnt nicht mit 0: {len(nicht_null)}")
    print(f"Duplikate (normalisiert): {dup_mask.sum()}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen)
Verwendete Spalte: 'Telefon'

1. Stichprobe (erste 10 Werte, Rohformat)
0       02372 8020
1    06221 / 98689
2    03834 / 22951
3    0162 / 249788
4    (07631) 67955
5      0235 530742
6    0271 / 300135
7    04251 / 96481
8    09392 / 18291
9    (0751) 787178

2. Fehlende Werte
Fehlende Telefonnummern: 0 von 1576

3. Erlaubte Zeichen prüfen
Nummern mit unerwarteten Zeichen: 0

4. Strukturmuster (Ziffern durch # ersetzt)
Anzahl unterschiedlicher Strukturmuster: 27
Telefon
##### / #####    573
#### / ######    264
(#####) #####    173
### / #######    122
##### #####       80
(####) ######     76
##### / ####      60
(###) #######     36
###### / ####     32
#### ######       31
#### / #####      22
(#####) ####      20
### / ######      16
### #######       13
(####) #####      10
(######) ####      8
##### ####         8
##### / ###        6
###### ####        6
###### / ###       5
#### / ####        4
#### ##### 

In [8]:
import re
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_H_Email.xlsx"
STICHWORT = "Nachname"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


def finde_spalte(df, stichwort):
    treffer = [c for c in df.columns if stichwort.lower() in c.lower()]
    if not treffer:
        raise KeyError(
            f"Keine Spalte gefunden, die '{stichwort}' enthält. "
            f"Vorhandene Spalten: {list(df.columns)}"
        )
    if len(treffer) > 1:
        print(f"WARNUNG: Mehrere Spalten passen zu '{stichwort}': {treffer}. Nehme die erste.")
    return treffer[0]


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    xlsx_path = finde_xlsx()
    df = pd.read_excel(xlsx_path)
    print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen)")

    col = finde_spalte(df, STICHWORT)
    print(f"Verwendete Spalte: {col!r}")

    section("1. Stichprobe (erste 10 Werte, Rohformat)")
    print(df[col].head(10).to_string())

    section("2. Fehlende Werte")
    fehlend = df[col].isna().sum()
    print(f"Fehlende Namen: {fehlend} von {len(df)}")

    section("3. Anzahl Kommas je Eintrag (erwartet: genau 1)")
    anzahl_kommas = df[col].astype(str).str.count(",")
    print(anzahl_kommas.value_counts().sort_index())

    ohne_komma = df[anzahl_kommas == 0]
    mehrere_kommas = df[anzahl_kommas > 1]
    print(f"\nOhne Komma (Format 'Nachname, Vorname' nicht erkennbar): {len(ohne_komma)}")
    if len(ohne_komma) > 0:
        print(ohne_komma[[col]].to_string(index=False))
    print(f"Mehr als ein Komma: {len(mehrere_kommas)}")
    if len(mehrere_kommas) > 0:
        print(mehrere_kommas[[col]].to_string(index=False))

    # ------------------------------------------------------------
    section("4. Nachname/Vorname aufteilen und auf leere Teile prüfen")
    split = df[col].astype(str).str.split(",", n=1, expand=True)
    nachname = split[0].str.strip()
    vorname = split[1].str.strip() if split.shape[1] > 1 else pd.Series([None] * len(df))

    leerer_nachname = df[nachname == ""]
    leerer_vorname = df[vorname.isna() | (vorname == "")]
    print(f"Leerer Nachname (vor dem Komma): {len(leerer_nachname)}")
    if len(leerer_nachname) > 0:
        print(leerer_nachname[[col]].to_string(index=False))
    print(f"Leerer Vorname (nach dem Komma): {len(leerer_vorname)}")
    if len(leerer_vorname) > 0:
        print(leerer_vorname[[col]].to_string(index=False))

    # ------------------------------------------------------------
    section("5. Unerwartete Leerzeichen (mehrfach, oder kein Leerzeichen nach Komma)")
    kein_leerzeichen_nach_komma = df[col].astype(str).str.contains(r",\S", regex=True, na=False)
    print(f"Kein Leerzeichen direkt nach dem Komma: {kein_leerzeichen_nach_komma.sum()}")
    if kein_leerzeichen_nach_komma.sum() > 0:
        print(df[kein_leerzeichen_nach_komma][[col]].head(15).to_string(index=False))

    doppelte_leerzeichen = df[col].astype(str).str.contains(r"  +", regex=True, na=False)
    print(f"Doppelte/mehrfache Leerzeichen: {doppelte_leerzeichen.sum()}")
    if doppelte_leerzeichen.sum() > 0:
        print(df[doppelte_leerzeichen][[col]].head(15).to_string(index=False))

    # ------------------------------------------------------------
    section("6. Ziffern oder unerwartete Sonderzeichen im Namen")
    # erlaubt: Buchstaben (inkl. Umlaute/Akzente), Leerzeichen, Bindestrich, Apostroph, Komma
    muster_erlaubt = re.compile(r"^[^\d]+$")
    mit_ziffern = df[~df[col].astype(str).apply(lambda v: bool(muster_erlaubt.match(v)))]
    print(f"Namen mit Ziffern: {len(mit_ziffern)}")
    if len(mit_ziffern) > 0:
        print(mit_ziffern[[col]].to_string(index=False))

    # ------------------------------------------------------------
    section("7. Duplikate (identischer Name)")
    dup = df[df.duplicated(subset=[col], keep=False)]
    print(f"Namen, die mehrfach vorkommen: {len(dup)}")
    if len(dup) > 0:
        print(dup[[col]].sort_values(col).to_string(index=False))

    # ------------------------------------------------------------
    section("8. Zusammenfassung")
    print(f"Zeilen gesamt: {len(df)}")
    print(f"Fehlend: {fehlend}")
    print(f"Ohne Komma: {len(ohne_komma)}")
    print(f"Mehrere Kommas: {len(mehrere_kommas)}")
    print(f"Leerer Nachname: {len(leerer_nachname)}")
    print(f"Leerer Vorname: {len(leerer_vorname)}")
    print(f"Kein Leerzeichen nach Komma: {kein_leerzeichen_nach_komma.sum()}")
    print(f"Doppelte Leerzeichen: {doppelte_leerzeichen.sum()}")
    print(f"Ziffern im Namen: {len(mit_ziffern)}")
    print(f"Duplikate: {len(dup)}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen)
Verwendete Spalte: 'Nachname, Vorname'

1. Stichprobe (erste 10 Werte, Rohformat)
0        Forster, Martin
1     Elina, Tsanaklidou
2          Vildan, şahin
3         Bäumker, Ellen
4     Bahadır, Bekiroğlu
5       Fink, Karl-Heinz
6           Gövert, Dirk
7           Rüsing, Paul
8        Wiesnewski, Tim
9    Stockbrink, Tatjana

2. Fehlende Werte
Fehlende Namen: 0 von 1576

3. Anzahl Kommas je Eintrag (erwartet: genau 1)
Nachname, Vorname
1    1576
Name: count, dtype: int64

Ohne Komma (Format 'Nachname, Vorname' nicht erkennbar): 0
Mehr als ein Komma: 0

4. Nachname/Vorname aufteilen und auf leere Teile prüfen
Leerer Nachname (vor dem Komma): 0
Leerer Vorname (nach dem Komma): 0

5. Unerwartete Leerzeichen (mehrfach, oder kein Leerzeichen nach Komma)
Kein Leerzeichen direkt nach dem Komma: 0
Doppelte/mehrfache Leerzeichen: 1
 Nachname, Vorname
Madalina  , Stoica

6. Ziffern oder unerwartete Sonderzeichen im Namen


In [9]:
import re
import pandas as pd
from pathlib import Path

FILENAME = "Let_Meet_H_Email.xlsx"
STICHWORT = "Straße"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_xlsx(dateiname=FILENAME):
    kandidaten = [
        SCRIPT_DIR / dateiname,
        SCRIPT_DIR.parent / dateiname,
        Path.cwd() / dateiname,
        Path.cwd().parent / dateiname,
        Path.home() / dateiname,
        Path.home() / "work" / dateiname,
        Path.home() / "LetsMeet" / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    orte = "\n".join(f"  - {k}" for k in kandidaten)
    raise FileNotFoundError(
        f"'{dateiname}' wurde an keinem der folgenden Orte gefunden:\n{orte}"
    )


def finde_spalte(df, stichwort):
    treffer = [c for c in df.columns if stichwort.lower() in c.lower()]
    if not treffer:
        raise KeyError(
            f"Keine Spalte gefunden, die '{stichwort}' enthält. "
            f"Vorhandene Spalten: {list(df.columns)}"
        )
    if len(treffer) > 1:
        print(f"WARNUNG: Mehrere Spalten passen zu '{stichwort}': {treffer}. Nehme die erste.")
    return treffer[0]


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    xlsx_path = finde_xlsx()
    df = pd.read_excel(xlsx_path)
    print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen)")

    col = finde_spalte(df, STICHWORT)
    print(f"Verwendete Spalte: {col!r}")
    name_col = "Nachname, Vorname" if "Nachname, Vorname" in df.columns else None

    def mit_name(sub_df):
        if name_col:
            return sub_df[[name_col, col]]
        return sub_df[[col]]

    section("1. Stichprobe (erste 10 Werte, Rohformat)")
    print(df[col].head(10).to_string())

    section("2. Fehlende Werte")
    fehlend = df[col].isna().sum()
    print(f"Fehlende Adressen: {fehlend} von {len(df)}")

    section("3. Anzahl Kommas je Eintrag (erwartet: genau 2)")
    anzahl_kommas = df[col].astype(str).str.count(",")
    print(anzahl_kommas.value_counts().sort_index())

    abweichend = df[anzahl_kommas != 2]
    print(f"\nAdressen mit abweichender Komma-Anzahl (nicht 2): {len(abweichend)}")
    if len(abweichend) > 0:
        print(mit_name(abweichend).to_string(index=False))

    # ------------------------------------------------------------
    section("4. Straße / PLZ / Ort aufteilen und auf leere Teile prüfen")
    split = df[col].astype(str).str.split(",", n=2, expand=True)
    strasse = split[0].str.strip() if split.shape[1] > 0 else pd.Series([None] * len(df))
    plz = split[1].str.strip() if split.shape[1] > 1 else pd.Series([None] * len(df))
    ort = split[2].str.strip() if split.shape[1] > 2 else pd.Series([None] * len(df))
    ort = ort.rename("Ort")

    for name, teil in [("Straße", strasse), ("PLZ", plz), ("Ort", ort)]:
        leer = df[teil.isna() | (teil == "")]
        print(f"Leer/fehlend: {name}: {len(leer)}")
        if len(leer) > 0:
            print(mit_name(leer).to_string(index=False))

    # ------------------------------------------------------------
    section("5. PLZ-Plausibilität (4-5-stellig, nur Ziffern)")
    plz_gueltig = plz.str.match(r"^\d{4,5}$", na=False)
    plz_ungueltig = df[~plz_gueltig]
    print(f"PLZ nicht 4-5-stellig numerisch: {len(plz_ungueltig)}")
    if len(plz_ungueltig) > 0:
        anzeige = mit_name(plz_ungueltig).copy()
        anzeige["PLZ_erkannt"] = plz[plz_ungueltig.index]
        print(anzeige.to_string(index=False))

    # führende Nullen prüfen (deutsche PLZ können mit 0 beginnen, z.B. Dresden 01xxx)
    plz_4stellig = plz[plz.str.match(r"^\d{4}$", na=False)]
    print(f"\n4-stellige PLZ (könnten fehlende führende Null haben): {len(plz_4stellig)}")
    if len(plz_4stellig) > 0:
        anzeige = df.loc[plz_4stellig.index]
        anzeige_out = mit_name(anzeige).copy()
        anzeige_out["PLZ"] = plz_4stellig
        print(anzeige_out.head(15).to_string(index=False))

    # ------------------------------------------------------------
    section("6. Straße enthält Hausnummer")
    hat_nummer = strasse.str.contains(r"\d", regex=True, na=False)
    ohne_nummer = df[~hat_nummer]
    print(f"Straße ohne erkennbare Hausnummer: {len(ohne_nummer)}")
    if len(ohne_nummer) > 0:
        anzeige = mit_name(ohne_nummer).copy()
        print(anzeige.to_string(index=False))

    # ------------------------------------------------------------
    section("7. Unerwartete Leerzeichen")
    doppelte_leerzeichen = df[col].astype(str).str.contains(r"  +", regex=True, na=False)
    print(f"Doppelte/mehrfache Leerzeichen: {doppelte_leerzeichen.sum()}")
    if doppelte_leerzeichen.sum() > 0:
        print(mit_name(df[doppelte_leerzeichen]).to_string(index=False))

    kein_leerzeichen_nach_komma = df[col].astype(str).str.contains(r",\S", regex=True, na=False)
    print(f"Kein Leerzeichen direkt nach einem Komma: {kein_leerzeichen_nach_komma.sum()}")
    if kein_leerzeichen_nach_komma.sum() > 0:
        print(mit_name(df[kein_leerzeichen_nach_komma]).head(15).to_string(index=False))

    # ------------------------------------------------------------
    section("8. Duplikate (identische Adresse)")
    dup = df[df.duplicated(subset=[col], keep=False)]
    print(f"Adressen, die mehrfach vorkommen: {len(dup)}")
    if len(dup) > 0:
        print(mit_name(dup).sort_values(col).to_string(index=False))

    # ------------------------------------------------------------
    section("9. Top 10 Orte")
    print(ort.value_counts().head(10))

    # ------------------------------------------------------------
    section("10. Zusammenfassung")
    print(f"Zeilen gesamt: {len(df)}")
    print(f"Fehlend: {fehlend}")
    print(f"Abweichende Komma-Anzahl: {len(abweichend)}")
    print(f"PLZ ungültig: {len(plz_ungueltig)}")
    print(f"4-stellige PLZ (Kontrolle empfohlen): {len(plz_4stellig)}")
    print(f"Straße ohne Hausnummer: {len(ohne_nummer)}")
    print(f"Doppelte Leerzeichen: {doppelte_leerzeichen.sum()}")
    print(f"Kein Leerzeichen nach Komma: {kein_leerzeichen_nach_komma.sum()}")
    print(f"Duplikate: {len(dup)}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen)
Verwendete Spalte: 'Straße Nr, PLZ Ort'

1. Stichprobe (erste 10 Werte, Rohformat)
0              Minslebener Str. 0, 46286, Dorsten
1                 Gartenweg 13, 69126, Heidelberg
2    Heinrich-Heine-Straße 99c, 17489, Greifswald
3                   Weidenring 39, 81539, München
4     Eichendorffstraße 20, 79379, Müllheim Baden
5                     Lindenweg 80, 44795, Bochum
6              Rahlenbeckstr. 100, 56070, Koblenz
7                 Im Brückle 44, 33334, Gütersloh
8                   Wilhelmsaue 29, 88131, Lindau
9            Isolde-Kurz-Str. 126C, 35394, Gießen

2. Fehlende Werte
Fehlende Adressen: 0 von 1576

3. Anzahl Kommas je Eintrag (erwartet: genau 2)
Straße Nr, PLZ Ort
2    1573
3       3
Name: count, dtype: int64

Adressen mit abweichender Komma-Anzahl (nicht 2): 3
Nachname, Vorname                              Straße Nr, PLZ Ort
    Lange, Ansgar       Boniverstr. 25, 17109, Demmin, Hansestadt

In [10]:
"""
LetsMeet - Finale Bereinigung (breites Format)
====================================================================
Liest Let_Meet_H_Email.xlsx ein, wendet folgende Regeln an und
speichert das Ergebnis als Let_Meet_HE_Breit.xlsx:

  1. Telefonnummer ohne führende '0' (User-Fehler: nationale Vorwahl
     vergessen) -> führende '0' wird ergänzt.
  2. Die 13 kürzeren Telefonnummern (<9 Ziffern) -> NICHT verändert,
     da sie existieren könnten.
  3. Name mit doppeltem Leerzeichen ('Madalina  , Stoica') -> wird
     auf ein Leerzeichen normalisiert.
  4. Doppelt vorkommender Name (z.B. 'Müller, Elisabeth' 2x) ->
     NICHT verändert, da real möglich.
  5. Vierstellige Postleitzahlen -> auf fünfstellig konvertiert
     (führende '0' ergänzt; Referenz/Verifikation z.B. via
     https://www.alte-postleitzahlen.de).
  6. Adressen mit abweichender Kommazahl ('Hansestadt'-Zusatz) ->
     'Hansestadt' wird entfernt.
  7. Spalte 'Nachname, Vorname' -> aufgeteilt in 'Nachname' / 'Vorname'.
  8. Spalte 'Straße Nr, PLZ Ort' -> aufgeteilt in 'Strasse Nr.' /
     'PLZ' / 'Ort'.
"""

import re
import pandas as pd
from pathlib import Path

INPUT_KANDIDATEN = ["Let_Meet_H_Email.xlsx", "Leets_Meet_H_Email.xlsx"]
OUTPUT_FILENAME = "Leets_Meet_HE_Breit.xlsx"

NAME_COL = "Nachname, Vorname"
ADRESSE_COL = "Straße Nr, PLZ Ort"
TELEFON_COL = "Telefon"

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()


def finde_datei(dateinamen):
    for dateiname in dateinamen:
        kandidaten = [
            SCRIPT_DIR / dateiname,
            SCRIPT_DIR.parent / dateiname,
            Path.cwd() / dateiname,
            Path.cwd().parent / dateiname,
            Path.home() / dateiname,
            Path.home() / "work" / dateiname,
            Path.home() / "LetsMeet" / dateiname,
        ]
        for pfad in kandidaten:
            if pfad.exists():
                return pfad
        treffer = list(Path.home().rglob(dateiname))
        if treffer:
            return treffer[0]
    orte = "\n".join(f"  - {d}" for d in dateinamen)
    raise FileNotFoundError(f"Keine der folgenden Dateien gefunden:\n{orte}")


def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def main():
    xlsx_path = finde_datei(INPUT_KANDIDATEN)
    df = pd.read_excel(xlsx_path)
    print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

    # ------------------------------------------------------------
    section("1+2. Telefonnummer: fehlende führende 0 ergänzen")
    # Erste Ziffer (nach evtl. Klammer) ermitteln
    erste_ziffer = df[TELEFON_COL].astype(str).str.strip().str.lstrip("(").str[0]
    fehlt_null = erste_ziffer != "0"
    anzahl_korrigiert = fehlt_null.sum()
    print(f"Telefonnummern ohne führende 0: {anzahl_korrigiert}")
    if anzahl_korrigiert > 0:
        print(df.loc[fehlt_null, ["Nachname, Vorname", TELEFON_COL]].to_string(index=False))

    df.loc[fehlt_null, TELEFON_COL] = "0" + df.loc[fehlt_null, TELEFON_COL].astype(str).str.lstrip()

    # Die 13 kürzeren Nummern (<9 Ziffern) bleiben bewusst unverändert -
    # keine Aktion nötig, da sie ohnehin nicht angefasst werden.
    anzahl_ziffern = df[TELEFON_COL].astype(str).apply(lambda t: len(re.sub(r"\D", "", t)))
    kurze_nummern = (anzahl_ziffern < 9).sum()
    print(f"Kurze Telefonnummern (<9 Ziffern, bleiben unverändert): {kurze_nummern}")

    # ------------------------------------------------------------
    section("3. Name: doppeltes Leerzeichen normalisieren")
    doppelt_leerzeichen = df[NAME_COL].astype(str).str.contains(r"  +", regex=True, na=False)
    print(f"Namen mit doppeltem Leerzeichen: {doppelt_leerzeichen.sum()}")
    if doppelt_leerzeichen.sum() > 0:
        print(df.loc[doppelt_leerzeichen, [NAME_COL]].to_string(index=False))

    df[NAME_COL] = df[NAME_COL].astype(str).str.replace(r"  +", " ", regex=True)

    # 4. Doppelte Namen -> bewusst nicht verändert (keine Aktion nötig)

    # ------------------------------------------------------------
    section("6. Adresse: 'Hansestadt' entfernen")
    hansestadt_mask = df[ADRESSE_COL].astype(str).str.contains("Hansestadt", na=False)
    print(f"Adressen mit 'Hansestadt': {hansestadt_mask.sum()}")
    if hansestadt_mask.sum() > 0:
        print(df.loc[hansestadt_mask, [ADRESSE_COL]].to_string(index=False))

    # ', Hansestadt' (mit Komma) und ' Hansestadt' (ohne Komma) beide abdecken,
    # anschließend übrig gebliebene doppelte Leerzeichen/Kommas aufräumen.
    df[ADRESSE_COL] = (
        df[ADRESSE_COL].astype(str)
        .str.replace(r",?\s*Hansestadt", "", regex=True)
        .str.rstrip()
    )

    print("\nNach Bereinigung:")
    print(df.loc[hansestadt_mask, [ADRESSE_COL]].to_string(index=False))

    # ------------------------------------------------------------
    section("5. Adresse: PLZ auf 5 Stellen bringen")
    split_check = df[ADRESSE_COL].str.split(",", n=2, expand=True)
    plz_check = split_check[1].str.strip()
    plz4_mask = plz_check.str.match(r"^\d{4}$", na=False)
    print(f"4-stellige PLZ (werden auf 5-stellig ergänzt): {plz4_mask.sum()}")
    if plz4_mask.sum() > 0:
        print(df.loc[plz4_mask, [ADRESSE_COL]].head(10).to_string(index=False))

    def plz_auffuellen(adresse):
        teile = adresse.split(",", 2)
        if len(teile) != 3:
            return adresse
        strasse, plz, ort = teile
        plz = plz.strip()
        if re.match(r"^\d{4}$", plz):
            plz = plz.zfill(5)
        return f"{strasse}, {plz}, {ort.strip()}"

    df[ADRESSE_COL] = df[ADRESSE_COL].apply(plz_auffuellen)

    # ------------------------------------------------------------
    section("7. Name aufteilen: Nachname / Vorname")
    name_split = df[NAME_COL].str.split(",", n=1, expand=True)
    df["Nachname"] = name_split[0].str.strip()
    df["Vorname"] = name_split[1].str.strip()

    # ------------------------------------------------------------
    section("8. Adresse aufteilen: Strasse Nr. / PLZ / Ort")
    adresse_split = df[ADRESSE_COL].str.split(",", n=2, expand=True)
    df["Strasse Nr."] = adresse_split[0].str.strip()
    df["PLZ"] = adresse_split[1].str.strip()
    df["Ort"] = adresse_split[2].str.strip()

    # Ursprüngliche zusammengesetzte Spalten entfernen, da jetzt aufgeteilt
    df = df.drop(columns=[NAME_COL, ADRESSE_COL])

    # Sinnvolle Spaltenreihenfolge
    neue_reihenfolge = ["Nachname", "Vorname", "Strasse Nr.", "PLZ", "Ort"] + [
        c for c in df.columns if c not in
        {"Nachname", "Vorname", "Strasse Nr.", "PLZ", "Ort"}
    ]
    df = df[neue_reihenfolge]

    # ------------------------------------------------------------
    section("Stichprobe des Ergebnisses (erste 5 Zeilen)")
    print(df[["Nachname", "Vorname", "Strasse Nr.", "PLZ", "Ort", TELEFON_COL]].head(5).to_string(index=False))

    # ------------------------------------------------------------
    out_path = xlsx_path.parent / OUTPUT_FILENAME
    df.to_excel(out_path, index=False)
    print(f"\nGespeichert: {out_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")
    print("HINWEIS: PLZ ist als Text mit führenden Nullen gespeichert. Beim erneuten "
          "Einlesen mit pandas bitte dtype={'PLZ': str} angeben, sonst werden führende "
          "Nullen beim Auto-Parsing entfernt (betrifft nur das Einlesen, nicht die Datei selbst).")

    # ------------------------------------------------------------
    section("Zusammenfassung")
    print(f"Telefonnummern mit ergänzter führender 0: {anzahl_korrigiert}")
    print(f"Kurze Telefonnummern (unverändert gelassen): {kurze_nummern}")
    print(f"Namen mit korrigiertem doppeltem Leerzeichen: {doppelt_leerzeichen.sum()}")
    print(f"Adressen mit entferntem 'Hansestadt': {hansestadt_mask.sum()}")
    print(f"PLZ auf 5 Stellen ergänzt: {plz4_mask.sum()}")


if __name__ == "__main__":
    main()

Eingelesen: /home/jovyan/LetsMeet/Let_Meet_H_Email.xlsx (1576 Zeilen, 18 Spalten)

1+2. Telefonnummer: fehlende führende 0 ergänzen
Telefonnummern ohne führende 0: 1
Nachname, Vorname    Telefon
  Zumdohme, Ernst  824696843
Kurze Telefonnummern (<9 Ziffern, bleiben unverändert): 13

3. Name: doppeltes Leerzeichen normalisieren
Namen mit doppeltem Leerzeichen: 1
 Nachname, Vorname
Madalina  , Stoica

6. Adresse: 'Hansestadt' entfernen
Adressen mit 'Hansestadt': 4
                             Straße Nr, PLZ Ort
 Brandenstein 106, 17489, Greifswald Hansestadt
      Boniverstr. 25, 17109, Demmin, Hansestadt
Borchshöher Str. 105, 17109, Demmin, Hansestadt
  Boltensternstr. 73, 17109, Demmin, Hansestadt

Nach Bereinigung:
                 Straße Nr, PLZ Ort
Brandenstein 106, 17489, Greifswald
      Boniverstr. 25, 17109, Demmin
Borchshöher Str. 105, 17109, Demmin
  Boltensternstr. 73, 17109, Demmin

5. Adresse: PLZ auf 5 Stellen bringen
4-stellige PLZ (werden auf 5-stellig ergänzt): 58
     

In [11]:
INPUT_FILENAME = "Leets_Meet_HE_Breit.xlsx"
OUTPUT_FILENAME = "Leets_Meet_HEB_Schuessel.xlsx"

def finde_datei(dateiname):
    kandidaten_basis = [Path.cwd(), Path.cwd().parent, Path.home(),
                         Path.home() / "work", Path.home() / "LetsMeet"]
    for basis in kandidaten_basis:
        pfad = basis / dateiname
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    raise FileNotFoundError(f"'{dateiname}' wurde nicht gefunden.")

xlsx_path = finde_datei(INPUT_FILENAME)
df = pd.read_excel(xlsx_path, dtype={'PLZ': str, 'Telefon': str})
print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")
df.insert(0, "install", range(1, len(df) + 1))
df.insert(1, "imp", "Excel")

print(f"Neue Spaltenanzahl: {df.shape[1]}")
df[["install", "imp"]].head(5)
print("'install' eindeutig fortlaufend:", list(df["install"]) == list(range(1, len(df) + 1)))
print("'imp' eindeutige Werte:", df["imp"].unique())
df.head(10)
out_path = xlsx_path.parent / OUTPUT_FILENAME
df.to_excel(out_path, index=False)
print(f"Gespeichert: {out_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

Eingelesen: /home/jovyan/LetsMeet/Leets_Meet_HE_Breit.xlsx (1576 Zeilen, 21 Spalten)
Neue Spaltenanzahl: 23
'install' eindeutig fortlaufend: True
'imp' eindeutige Werte: ['Excel']
Gespeichert: /home/jovyan/LetsMeet/Leets_Meet_HEB_Schuessel.xlsx (1576 Zeilen, 23 Spalten)


In [12]:
SAMMELSPALTE = "Hobby1 %Prio1%; Hobby2 %Prio2%; Hobby3 %Prio3%; Hobby4 %Prio4%; Hobby5 %Prio5%;"
if SAMMELSPALTE in df.columns:
    df = df.drop(columns=[SAMMELSPALTE])
    print("Sammelspalte entfernt.")
else:
    print("Sammelspalte war nicht vorhanden.")
print(f"Spaltenanzahl: {df.shape[1]}")

OUTPUT_FILENAME_HEBS = "Leets_Meet_HEBS.xlsx"
out_path = xlsx_path.parent / OUTPUT_FILENAME_HEBS
df.to_excel(out_path, index=False)
print(f"Gespeichert: {out_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

Sammelspalte entfernt.
Spaltenanzahl: 22
Gespeichert: /home/jovyan/LetsMeet/Leets_Meet_HEBS.xlsx (1576 Zeilen, 22 Spalten)


In [13]:
import sys
!{sys.executable} -m pip install psycopg2-binary --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [14]:
import sys
!{sys.executable} -m pip install sqlalchemy psycopg2-binary --break-system-packages -q

import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

FILENAME = "Leets_Meet_HEBS.xlsx"
TABLE_NAME = "letsmeet"

DB_USER = "user"
DB_PASSWORD = "secret"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "lf8_lets_meet_db"

def finde_xlsx(dateiname=FILENAME):
    kandidaten_basis = [Path.cwd(), Path.cwd().parent, Path.home(),
                         Path.home() / "work", Path.home() / "LetsMeet"]
    for basis in kandidaten_basis:
        pfad = basis / dateiname
        if pfad.exists():
            return pfad
    treffer = list(Path.home().rglob(dateiname))
    if treffer:
        return treffer[0]
    raise FileNotFoundError(f"'{dateiname}' wurde nicht gefunden.")

xlsx_path = finde_xlsx()

# PLZ und Telefon als Text einlesen, damit fuehrende Nullen erhalten bleiben
df = pd.read_excel(xlsx_path, dtype={"PLZ": str, "Telefon": str})
print(f"Eingelesen: {xlsx_path} ({df.shape[0]} Zeilen, {df.shape[1]} Spalten)")

# Geburtsdatum von Text (TT.MM.JJJJ) in echtes Datum umwandeln
df["Geburtsdatum"] = pd.to_datetime(df["Geburtsdatum"], format="%d.%m.%Y", errors="coerce").dt.date
nicht_parsebar = df["Geburtsdatum"].isna().sum()
if nicht_parsebar > 0:
    print(f"WARNUNG: {nicht_parsebar} Geburtsdaten konnten nicht geparst werden.")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print(f"Verbunden mit: {version}")

# Tabelle mit expliziten Datentypen anlegen:
#  - install: SERIAL (echter auto-increment Typ)
#  - PLZ: TEXT (bewusst NICHT numerisch, trotz Ziffern-Erscheinungsbild)
#  - Prio1..Prio5: INTEGER
#  - Geburtsdatum: DATE
#  - alles andere: TEXT
with engine.begin() as conn:
    conn.execute(text(f'DROP TABLE IF EXISTS "{TABLE_NAME}"'))
    conn.execute(text(f"""
        CREATE TABLE "{TABLE_NAME}" (
            install SERIAL,
            imp TEXT,
            "Nachname" TEXT,
            "Vorname" TEXT,
            "Strasse Nr." TEXT,
            "PLZ" TEXT,
            "Ort" TEXT,
            "Telefon" TEXT,
            "E-Mail" TEXT,
            "Geschlecht (m/w/nonbinary)" TEXT,
            "Interessiert an" TEXT,
            "Geburtsdatum" DATE,
            "Hobby1" TEXT, "Prio1" INTEGER,
            "Hobby2" TEXT, "Prio2" INTEGER,
            "Hobby3" TEXT, "Prio3" INTEGER,
            "Hobby4" TEXT, "Prio4" INTEGER,
            "Hobby5" TEXT, "Prio5" INTEGER,
            PRIMARY KEY ("install", "imp")
        )
    """))
print(f"Tabelle '{TABLE_NAME}' mit expliziten Datentypen angelegt.")

# Daten einfuegen
df.to_sql(TABLE_NAME, engine, if_exists="append", index=False)
print(f"{len(df)} Zeilen eingefuegt.")

# SERIAL-Sequenz auf den aktuellen Hoechstwert nachziehen,
# da wir install-Werte explizit mitgeliefert haben
with engine.begin() as conn:
    conn.execute(text(f"""
        SELECT setval(
            pg_get_serial_sequence('"{TABLE_NAME}"', 'install'),
            (SELECT MAX(install) FROM "{TABLE_NAME}")
        )
    """))
print("SERIAL-Sequenz von 'install' nachgezogen.")

# Kontrolle
with engine.connect() as conn:
    anzahl = conn.execute(text(f'SELECT COUNT(*) FROM "{TABLE_NAME}"')).scalar()
    print(f"\nKontrolle: {anzahl} Zeilen in der Tabelle '{TABLE_NAME}' (Excel hatte {len(df)}).")
    stichprobe = conn.execute(text(f'SELECT * FROM "{TABLE_NAME}" LIMIT 3')).fetchall()
    print("\nStichprobe (erste 3 Zeilen aus der DB):")
    for zeile in stichprobe:
        print(zeile)

Eingelesen: /home/jovyan/LetsMeet/Leets_Meet_HEBS.xlsx (1576 Zeilen, 22 Spalten)
Verbunden mit: PostgreSQL 16.14 on powerpc64le-conda-linux-gnu, compiled by powerpc64le-conda-linux-gnu-cc (conda-forge gcc 14.3.0-19) 14.3.0, 64-bit
Tabelle 'letsmeet' mit expliziten Datentypen angelegt.
1576 Zeilen eingefuegt.
SERIAL-Sequenz von 'install' nachgezogen.

Kontrolle: 1576 Zeilen in der Tabelle 'letsmeet' (Excel hatte 1576).

Stichprobe (erste 3 Zeilen aus der DB):
(1, 'Excel', 'Forster', 'Martin', 'Minslebener Str. 0', '46286', 'Dorsten', '02372 8020', 'martin.forster@web.org', 'm', 'w', datetime.date(1959, 3, 7), 'Fremdsprachenkenntnisse erweitern', 78, 'Im Wasser waten', 80, 'Schwierige Probleme klären', 61, 'Morgens Früh aufstehen', 17, None, None)
(2, 'Excel', 'Elina', 'Tsanaklidou', 'Gartenweg 13', '69126', 'Heidelberg', '06221 / 98689', 'tsanaklidou.elina@1und1.de', 'w', 'm', datetime.date(1958, 2, 28), 'Für jemanden kochen', 57, 'Mir die Probleme von anderen anhören', 21, 'Abends sein

In [17]:
with engine.begin() as conn:
    conn.execute(text(f'DROP VIEW IF EXISTS migration_users'))
    conn.execute(text(f"""
        CREATE VIEW migration_users AS
        SELECT
            "E-Mail"      AS email,
            "Vorname"     AS first_name,
            "Nachname"    AS last_name,
            "Geburtsdatum" AS birth_date,
            "PLZ"         AS postal_code,
            "Ort"         AS city
        FROM "{TABLE_NAME}"
    """))
print("View 'migration_users' angelegt.")

# Kontrolle: Spaltennamen und -typen der View pruefen
with engine.connect() as conn:
    spalten = conn.execute(text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'migration_users'
        ORDER BY ordinal_position
    """)).fetchall()
    print("\nSpalten der View 'migration_users':")
    for spalte in spalten:
        print(f"  {spalte[0]}: {spalte[1]}")

    anzahl = conn.execute(text("SELECT COUNT(*) FROM migration_users")).scalar()
    print(f"\nAnzahl Zeilen: {anzahl}")

    stichprobe = conn.execute(text("SELECT * FROM migration_users LIMIT 3")).fetchall()
    print("\nStichprobe:")
    for zeile in stichprobe:
        print(zeile)

View 'migration_users' angelegt.

Spalten der View 'migration_users':
  email: text
  first_name: text
  last_name: text
  birth_date: date
  postal_code: text
  city: text

Anzahl Zeilen: 1576

Stichprobe:
('martin.forster@web.org', 'Martin', 'Forster', datetime.date(1959, 3, 7), '46286', 'Dorsten')
('tsanaklidou.elina@1und1.de', 'Tsanaklidou', 'Elina', datetime.date(1958, 2, 28), '69126', 'Heidelberg')
('şahin.vildan@gmail.org', 'şahin', 'Vildan', datetime.date(1996, 7, 27), '17489', 'Greifswald')
